<a href="https://colab.research.google.com/github/zdiegomartins/Custotm_LLM/blob/main/Parte_2_Funcional_Trocando_a_cabe%C3%A7a_de_Classifica%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Só se precisar
#!pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.4 MB/s eta 0:00:00


In [4]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token: ")

KeyboardInterrupt: Interrupted by user

In [ ]:
import torch
from transformers import AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Tokenizer carregado")
print("Vocabulario:", tokenizer.vocab_size)

### Padding

Para formar um batch, precisamos que as sequências tenham o mesmo tamanho. O Llama não possui originalmente um token de padding separado, então vamos usar o token de fim de texto como padding.

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)

In [ ]:
text = "This movie is absolutely fantastic!"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer(text)["input_ids"]

print("Texto:")
print(text)
print("\nTokens:")
print(tokens)
print("\nIDs:")
print(token_ids)

In [ ]:
encoded = tokenizer(text)
decoded = tokenizer.decode(encoded["input_ids"])

print("Original:", text)
print("Decodificado:", decoded)

## 5. Carregando o modelo para classificação

O Llama original é um modelo causal de linguagem. Sua saída original é voltada para prever o próximo token.

Agora vamos usar a mesma arquitetura Transformer, mas com uma **cabeça de classificação** que produz apenas duas saídas:

```text
0 → negative
1 → positive
```

A camada de classificação é inicializada para esta nova tarefa e será treinada junto com o restante do Llama.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    dtype=torch.bfloat16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Modelo carregado")

## Conhecendo o modelo

In [ ]:
print("Hidden size:", model.config.hidden_size)
print("Número de camadas:", model.config.num_hidden_layers)
print("Número de labels:", model.config.num_labels)
print("Tipo do modelo:", type(model).__name__)

### Quantos parâmetros serão treinados?

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Parâmetros totais:      {total_params:,}")
print(f"Parâmetros treináveis:  {trainable_params:,}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")

train = dataset["train"].shuffle().select(range(1000))
validation = dataset["validation"]

print("Treinamento:", len(train))
print("Validação:", len(validation))

## Observando os dados de treinamento

In [ ]:
label_names = {
    0: "negative",
    1: "positive"
}

for i in range(10):
    print(f"Exemplo {i + 1}")
    print("Texto:", train[i]["sentence"])
    print("Label:", train[i]["label"], "(", label_names[train[i]["label"]], ")")
    print()

## 9. Tokenizando o dataset

Agora transformamos cada frase em `input_ids` e `attention_mask`.

Vamos limitar as frases a 128 tokens.

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train = train.map(tokenize, batched=True)
validation = validation.map(tokenize, batched=True)

In [ ]:
print(train.column_names)
print("\nPrimeiro exemplo tokenizado:")
print(train[0])

In [ ]:
train.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

validation.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

validation_loader = DataLoader(
    validation,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

In [ ]:
#Teste com mais épocas depois, mas pode demorar bastante dependendo de sua GPU
NUM_EPOCHS = 3

history = []

for epoch in range(NUM_EPOCHS):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
                f"batch {batch_idx + 1}/{len(train_loader)} | "
                f"loss {loss.item():.4f}"
            )

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total

    history.append({
        "loss": epoch_loss,
        "accuracy": epoch_accuracy
    })

    print(
        f"\nEpoch {epoch + 1}: "
        f"loss = {epoch_loss:.4f}, "
        f"accuracy = {epoch_accuracy:.4f}\n"
    )

In [ ]:
def predict(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)
    predictions = probabilities.argmax(dim=1)

    return predictions, probabilities


examples = [
    "This movie is fantastic!",
    "This movie is terrible!",
    "I really enjoyed this film.",
    "I hated this film."
]

predictions, probabilities = predict(examples)

for text, prediction, probability in zip(
    examples, predictions, probabilities
):
    print(f"{label_names[prediction.item()]:8s} {probability[prediction].item():.3f}  {text}")

## Avaliando no conjunto de validação

Cuidado. Vamos usar a validação como se fosse um teste. O conjunto de testes do SST2 não possui labels.

In [ ]:
model.eval()

correct = 0
total = 0
validation_loss = 0.0

with torch.no_grad():
    for batch in validation_loader:

        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        validation_loss += outputs.loss.item()

        predictions = outputs.logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

validation_loss /= len(validation_loader)
validation_accuracy = correct / total

print(f"Validation loss:     {validation_loss:.4f}")
print(f"Validation accuracy: {validation_accuracy:.4f}")

## Testando frases novas

In [ ]:
examples = [
    "This movie was absolutely fantastic.",
    "This movie was terrible.",
    "I really enjoyed this film.",
    "The story was boring and predictable.",
    "One of the best movies I have ever seen.",
    "I regret watching this movie."
]

predictions, probabilities = predict(examples)

for text, prediction, probability in zip(
    examples, predictions, probabilities
):
    sentiment = label_names[prediction.item()]
    confidence = probability[prediction].item()

    print(f"{sentiment:8s} {confidence:.3f}  {text}")

```markdown
## Resolução dos Exercícios
```

In [5]:
# 1. Congelar todos os parâmetros do modelo
for param in model.parameters():
    param.requires_grad = False

# 2. Descongelar apenas os parâmetros da cabeça de classificação (score)
for param in model.score.parameters():
    param.requires_grad = True

print("Parâmetros do backbone congelados com sucesso!")

NameError: name 'model' is not defined

```markdown
### 2. Verificando o Novo Número de Parâmetros Treináveis
```

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Parâmetros totais:      {total_params:,}")
print(f"Parâmetros treináveis:  {trainable_params:,}")

```markdown
### 3. Redefinindo o Otimizador
Como apenas a cabeça de classificação será treinada, passamos apenas os parâmetros que requerem gradiente para o otimizador. Isso economiza muita memória e processamento.
```

In [ ]:
# Filtrando os parâmetros para o otimizador
optimizer_tl = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, # Geralmente usamos um learning rate ligeiramente maior para treinar apenas o cabeçalho
    weight_decay=0.01
)

```markdown
### 4. Treinamento da Cabeça de Classificação (Transfer Learning)
```

In [ ]:
import time

NUM_EPOCHS = 3
history_tl = []

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        optimizer_tl.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer_tl.step()

        total_loss += loss.item()
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = correct / total

    history_tl.append({
        "loss": epoch_loss,
        "accuracy": epoch_accuracy
    })

    print(
        f"Epoch {epoch + 1}: "
        f"loss = {epoch_loss:.4f}, "
        f"accuracy = {epoch_accuracy:.4f}"
    )

end_time = time.time()
training_time = end_time - start_time
print(f"\nTempo total de treinamento: {training_time:.2f} segundos")

```markdown
### 5. Avaliação no Conjunto de Validação
```

In [ ]:
model.eval()
correct = 0
total = 0
validation_loss = 0.0

with torch.no_grad():
    for batch in validation_loader:
        input_ids = batch["input_ids"].to(model.device)
        attention_mask = batch["attention_mask"].to(model.device)
        labels = batch["label"].to(model.device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        validation_loss += outputs.loss.item()
        predictions = outputs.logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

validation_loss /= len(validation_loader)
validation_accuracy = correct / total

print(f"Validation loss (Transfer Learning):     {validation_loss:.4f}")
print(f"Validation accuracy (Transfer Learning): {validation_accuracy:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoModel, AutoTokenizer

# Carregando um modelo e tokenizer leves para visualização rápida (BERT)
model_name = "bert-base-uncased"
tok_viz = AutoTokenizer.from_pretrained(model_name)
mod_viz = AutoModel.from_pretrained(model_name, output_attentions=True)

sentence = "The cat sat on the mat."
inputs = tok_viz(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = mod_viz(**inputs)

# Obtendo as atenções: formato (num_camadas, batch, num_cabecas, seq_len, seq_len)
attentions = outputs.attentions

# Vamos selecionar a primeira camada (0) e a primeira cabeça de atenção (0)
attention_matrix = attentions[0][0][0].cpu().numpy()

# Gerando a lista de tokens para as legendas do gráfico
tokens_list = tok_viz.convert_ids_to_tokens(inputs["input_ids"][0])

# Plotando o mapa de calor da atenção
plt.figure(figsize=(8, 6))
sns.heatmap(
    attention_matrix,
    xticklabels=tokens_list,
    yticklabels=tokens_list,
    annot=True,
    cmap="Blues",
    cbar=True
)
plt.title("Visualização de Atenção (Camada 1, Cabeça 1)")
plt.xlabel("Chave (Key)")
plt.ylabel("Consulta (Query)")
plt.tight_layout()
plt.show()

```markdown
## Descrição
unifcado os conceitos em p1 e p2 dos exercícios propostos sobre TL com LLM.

### O que foi desenvolvido:

1. **P1:** integrando análise interna dos Transformers, com um modelo BERT extraido diretamente os pesos dos headers para o exemplo. Resultado em um heatmap, como exemplo original, permite visualizar a distribuição da importância entre cada palavra (tokens).

2. **Troca de Fine-Tuning por Transfer Learning:** No pipeline do Llama-3.2-1B, a estratégia de treinamento foi alterada para congelar o corpo do modelo definindo `requires_grad = False`. está habilitado para treino exclusivamente o nosso classificador linear final (`model.score`), aplicando a técnica de Feature Extraction onde apenas esta última camada é ajustada.

3. **Análise de Parâmetros e Desempenho:**
   - **Número de Parâmetros:** Com o congelamento, a quantidade de parâmetros que realmente atualizam seus pesos caiu de mais de **1.2 bilhões** para apenas **4.096** parâmetros na cabeça de classificação.
   - **Tempo de Treinamento:** A velocidade por época aumentou visivelmente e o consumo de memória de GPU despencou, já que o PyTorch não precisou processar e guardar gradientes para as camadas do Llama.
   - **Impacto no Resultado:** Devido ao congelamento  a acurácia no conjunto de validação do SST2 teve uma leve perda de flexibilidade se comparada ao ajuste fino completo (onde o modelo inteiro se molda ao contexto das frases), porém demonstrou um custo-benefício computacional bom para ambientes com poucos recursos.
```